# Waste AI v4 — Fine-tuning ciblé Paper & Glass

Ce notebook part du checkpoint **waste_ai_v3.pt** (EfficientNet-B2) et affine le modèle
pour améliorer les classes **papier** et **verre**, qui sont les plus confondues.

## Avant de commencer
1. Active **GPU T4** : Settings → Accelerator → GPU T4 x2
2. Active **Internet** : Settings → Internet → On
3. Ajoute ton dataset `waste-ai-v3` en input (contenant `waste_ai_v3.pt`)
   - Si pas encore fait : sur Kaggle → New Dataset → upload `waste_ai_v3.pt` → nom : `waste-ai-v3`
   - Dans ce notebook : + Add Input → recherche `waste-ai-v3` → Add

Exécute les cellules dans l'ordre.

## Étape 1 — Installation

In [ ]:
# Sur Kaggle, torch/torchvision sont pré-installés — on installe seulement le reste
!pip install timm albumentations tqdm -q

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE != 'cuda':
    print('ATTENTION : Active le GPU T4 dans Settings → Accelerator')

## Étape 2 — Téléchargement TrashNet

In [ ]:
import os

TRASHNET_DIR = '/kaggle/working/trashnet'

if not os.path.exists(f'{TRASHNET_DIR}/dataset-resized'):
    print('Téléchargement TrashNet...')
    !wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip -O /kaggle/working/dataset-resized.zip
    !unzip -q /kaggle/working/dataset-resized.zip -d {TRASHNET_DIR}/
    print('TrashNet téléchargé')
else:
    print('TrashNet déjà présent')

classes = [c for c in os.listdir(f'{TRASHNET_DIR}/dataset-resized') if not c.startswith('.')]
print('Classes TrashNet :', sorted(classes))
for c in sorted(classes):
    n = len(os.listdir(f'{TRASHNET_DIR}/dataset-resized/{c}'))
    print(f'  {c}: {n} images')

## Étape 3 — Téléchargement TACO (URLs directes)

In [ ]:
import os, json, requests
from tqdm import tqdm

TACO_DIR = '/kaggle/working/TACO'
os.makedirs(f'{TACO_DIR}/data/images', exist_ok=True)

# Téléchargement des annotations
ann_path = f'{TACO_DIR}/data/annotations.json'
if not os.path.exists(ann_path):
    print('Téléchargement annotations TACO...')
    url = 'https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json'
    r = requests.get(url, timeout=30)
    with open(ann_path, 'w') as f:
        f.write(r.text)
    print('Annotations téléchargées')

with open(ann_path) as f:
    taco_data = json.load(f)

print(f'TACO : {len(taco_data["images"])} images dans les annotations')

# Télécharger les images manquantes
downloaded = 0
errors = 0
for img in tqdm(taco_data['images'], desc='Téléchargement images TACO'):
    img_id = img['id']
    out_path = f'{TACO_DIR}/data/images/{img_id}.jpg'
    if os.path.exists(out_path):
        continue
    url = img.get('flickr_url') or img.get('coco_url', '')
    if not url:
        continue
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            with open(out_path, 'wb') as f:
                f.write(r.content)
            downloaded += 1
    except:
        errors += 1

total_imgs = len([f for f in os.listdir(f'{TACO_DIR}/data/images') if f.endswith('.jpg')])
print(f'Téléchargées : {downloaded} | Erreurs : {errors} | Total sur disque : {total_imgs}')

## Étape 4 — Organisation TACO par classe

In [ ]:
import os, json, shutil

TACO_DIR = '/kaggle/working/TACO'
TACO_OUT = '/kaggle/working/taco_organized'
CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

TACO_TO_CLASS = {
    'Plastic bottle': 'plastic', 'Plastic bag & wrapper': 'plastic',
    'Plastic container': 'plastic', 'Plastic cup': 'plastic',
    'Plastic lid': 'plastic', 'Plastic straw': 'plastic',
    'Plastic utensils': 'plastic', 'Six pack rings': 'plastic',
    'Styrofoam piece': 'plastic',
    'Paper': 'paper', 'Paper bag': 'paper', 'Newspaper': 'paper',
    'Cardboard': 'cardboard', 'Carton': 'cardboard', 'Drink carton': 'cardboard',
    'Glass bottle': 'glass', 'Broken glass': 'glass', 'Glass jar': 'glass',
    'Aluminium foil': 'metal', 'Metal bottle cap': 'metal',
    'Drink can': 'metal', 'Food can': 'metal', 'Pop tab': 'metal',
    'Scrap metal': 'metal',
    'Cigarette': 'trash', 'Rope & strings': 'trash',
    'Shoe': 'trash', 'Unlabeled litter': 'trash',
}

for cls in CLASS_NAMES:
    os.makedirs(f'{TACO_OUT}/{cls}', exist_ok=True)

with open(f'{TACO_DIR}/data/annotations.json') as f:
    taco_data = json.load(f)

id_to_file = {img['id']: img['id'] for img in taco_data['images']}
cat_id_to_name = {cat['id']: cat['name'] for cat in taco_data['categories']}

count = 0
for ann in taco_data['annotations']:
    cat_name = cat_id_to_name.get(ann['category_id'], '')
    target_class = TACO_TO_CLASS.get(cat_name)
    if target_class is None:
        continue
    img_id = ann['image_id']
    src = f"{TACO_DIR}/data/images/{img_id}.jpg"
    dst = f"{TACO_OUT}/{target_class}/{img_id}_{ann['id']}.jpg"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        count += 1

print(f'TACO : {count} images organisées')
for cls in CLASS_NAMES:
    n = len(os.listdir(f'{TACO_OUT}/{cls}'))
    print(f'  {cls}: {n} images')

## Étape 5 — Chargement du modèle v3

In [ ]:
import torch
import timm
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
NUM_CLASSES = len(CLASS_NAMES)

# Cherche waste_ai_v3.pt dans les inputs Kaggle
V3_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'waste_ai_v3.pt':
            V3_PATH = os.path.join(root, f)
            break

if V3_PATH is None:
    raise FileNotFoundError(
        'waste_ai_v3.pt introuvable !\n'
        'Ajoute ton dataset Kaggle via + Add Input dans ce notebook.'
    )

print(f'Modèle v3 trouvé : {V3_PATH}')

model = timm.create_model('efficientnet_b2', pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(V3_PATH, map_location='cpu'))
model = model.to(DEVICE)
print(f'Modèle chargé sur {DEVICE}')

## Étape 6 — Dataset avec surreprésentation Paper & Glass

In [ ]:
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split, WeightedRandomSampler
from torchvision import datasets
import albumentations as A
from albumentations.pytorch import ToTensorV2

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
TRASHNET_DIR = '/kaggle/working/trashnet'
TACO_OUT = '/kaggle/working/taco_organized'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Augmentation plus forte pour paper/glass
strong_transform = A.Compose([
    A.RandomResizedCrop(size=(260, 260), scale=(0.4, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=45, p=0.6),
    A.OneOf([
        A.GaussNoise(std_range=(0.05, 0.2)),
        A.GaussianBlur(blur_limit=5),
        A.MotionBlur(blur_limit=5),
    ], p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4),
        A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40),
        A.CLAHE(clip_limit=6),
    ], p=0.6),
    A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(8, 40), hole_width_range=(8, 40), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=260, width=260),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


class WasteDataset(Dataset):
    def __init__(self, root, transform=None):
        self.data = datasets.ImageFolder(root)
        self.transform = transform
        self.classes = self.data.classes
        self.targets = [s[1] for s in self.data.samples]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data.samples[idx]
        image = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label


trashnet_ds = WasteDataset(f'{TRASHNET_DIR}/dataset-resized', transform=strong_transform)
taco_ds = WasteDataset(TACO_OUT, transform=strong_transform)

print(f'TrashNet classes : {trashnet_ds.classes}')
print(f'TACO classes     : {taco_ds.classes}')

full_dataset = ConcatDataset([trashnet_ds, taco_ds])

# Split train/val
n_train = int(0.8 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_set, val_set = random_split(full_dataset, [n_train, n_val],
                                   generator=torch.Generator().manual_seed(42))

# --- WeightedRandomSampler : surreprésentation paper (idx 3) et glass (idx 1) ---
# Mapping class_name → index dans ImageFolder (ordre alphabétique)
# cardboard=0, glass=1, metal=2, paper=3, plastic=4, trash=5
CLASS_WEIGHTS_SAMPLE = {
    0: 1.0,   # cardboard
    1: 3.0,   # glass  ← x3
    2: 1.0,   # metal
    3: 3.0,   # paper  ← x3
    4: 1.0,   # plastic
    5: 1.0,   # trash
}

# Récupère les labels du train_set
all_targets_trashnet = trashnet_ds.targets
all_targets_taco = taco_ds.targets
all_targets = all_targets_trashnet + all_targets_taco

train_indices = train_set.indices
train_targets = [all_targets[i] for i in train_indices]
sample_weights = [CLASS_WEIGHTS_SAMPLE[t] for t in train_targets]

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_set, batch_size=24, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=24, shuffle=False,  num_workers=2, pin_memory=True)

print(f'Dataset total : {len(full_dataset)} | Train : {n_train} | Val : {n_val}')
print('WeightedRandomSampler actif — paper et glass surreprésentés x3')

## Étape 7 — Fine-tuning ciblé (loss pondérée, LR très bas)

In [ ]:
import torch
import torch.nn as nn
import os

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

# Loss pondérée : pénalise plus les erreurs sur paper et glass
# cardboard=0, glass=1, metal=2, paper=3, plastic=4, trash=5
class_weights = torch.tensor([1.0, 2.5, 1.0, 2.5, 1.0, 1.0]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# LR différentiel : très bas pour ne pas détruire les acquis du v3
# Backbone → LR ultra-faible | Tête → LR légèrement plus élevé
optimizer = torch.optim.AdamW([
    {'params': model.blocks.parameters(), 'lr': 5e-6},
    {'params': model.conv_stem.parameters(), 'lr': 5e-6},
    {'params': model.classifier.parameters(), 'lr': 2e-5},
], weight_decay=1e-4)

EPOCHS = 12
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_acc': []}
best_acc = 0.0


def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total * 100


print(f'Fine-tuning v3 → v4 sur {EPOCHS} epochs')
print('Classes cibles : glass (x2.5 loss) et paper (x2.5 loss) | sampling x3\n')

for epoch in range(EPOCHS):
    loss, _ = run_epoch(train_loader, train=True)
    _, acc  = run_epoch(val_loader,   train=False)
    scheduler.step()
    history['train_loss'].append(loss)
    history['val_acc'].append(acc)
    tag = ''
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), '/kaggle/working/checkpoints/waste_ai_v4.pt')
        tag = ' ✓ sauvegardé'
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {loss:.4f} | Val Acc: {acc:.1f}%{tag}')

print(f'\nMeilleure accuracy : {best_acc:.1f}%')
print('Fichier sauvegardé : /kaggle/working/checkpoints/waste_ai_v4.pt')

## Étape 8 — Évaluation par classe (focus Paper & Glass)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import torch

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model.load_state_dict(torch.load('/kaggle/working/checkpoints/waste_ai_v4.pt', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print('=== Rapport de classification ===')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Courbes d'entraînement
axes[0].plot(history['train_loss'], color='#e74c3c', label='Loss')
axes[0].set_title('Loss — Fine-tuning v4')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

ax2 = axes[0].twinx()
ax2.plot(history['val_acc'], color='#2ecc71', linestyle='--', label='Val Acc')
ax2.set_ylabel('Accuracy (%)')
ax2.legend(loc='upper right')

# Matrice de confusion
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Matrice de confusion — Waste AI v4')
axes[1].set_ylabel('Réel')
axes[1].set_xlabel('Prédit')

plt.suptitle('Waste AI v4 — Fine-tuning Paper & Glass', fontsize=14)
plt.tight_layout()
plt.savefig('/kaggle/working/checkpoints/eval_v4.png', dpi=150)
plt.show()

# Focus paper & glass
print('\n=== Focus Paper & Glass ===')
for cls_idx, cls_name in [(1, 'glass'), (3, 'paper')]:
    tp = cm[cls_idx, cls_idx]
    total_real = cm[cls_idx].sum()
    recall = tp / total_real * 100 if total_real > 0 else 0
    print(f'{cls_name:10s} : {tp}/{total_real} correctement classés → Recall = {recall:.1f}%')

## Étape 9 — Télécharger le modèle v4

**Sur Kaggle, pas besoin de code pour télécharger.**

1. Clique sur l'onglet **Output** (panneau de droite)
2. Navigue dans `checkpoints/`
3. Clique sur le fichier → **Download**

Si tu ne vois pas `checkpoints/`, exécute la cellule ci-dessous pour copier le fichier à la racine.

In [ ]:
import shutil
shutil.copy('/kaggle/working/checkpoints/waste_ai_v4.pt', '/kaggle/working/waste_ai_v4.pt')
print('waste_ai_v4.pt copié à la racine de /kaggle/working/')
print('-> Rafraîchis le panneau Output pour le voir et le télécharger')
print('-> Place-le dans waste-ai/model/checkpoints/ sur ton PC')
print('-> L API chargera automatiquement la v4 (renomme-le waste_ai_v3.pt ou ajoute la v4 dans main.py)')